# Experimento de Embeddings & Teste de Busca Vetorial

Objetivos deste notebook:
1. Carregar os chunks processados (`data/processed/chunks.parquet`).
2. Verificar a GPU (NVIDIA RTX 3060) e instanciar o modelo de Embedding via `sentence-transformers`.
3. Gerar embeddings para uma amostra e inspecionar os vetores gerados.
4. Criar uma coleção no **ChromaDB** e testar uma busca por similaridade real.

# 1.Setup

In [1]:
import torch
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from ri_chat.config import PROCESSED_DATA_DIR, MODELS_DIR

2026-11-11 10:18:17.760 | INFO     | ri_chat.config:<module>:11 - PROJ_ROOT path is: /home/gabriel_santiago/Estudos/Portfolio/ri_chat


In [2]:
# checando hardware
cuda_ok = torch.cuda.is_available()
device = "cuda" if cuda_ok else "cpu"
print(f"CUDA Disponível: {cuda_ok}")
if cuda_ok:
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

CUDA Disponível: True
Dispositivo: NVIDIA GeForce RTX 3060
VRAM Total: 12.88 GB


# 2.Testes

## 2.1.Embedding e Busca por Similaridade

In [3]:
# carregando  o modelo escolhido
MODEL_NAME = "BAAI/bge-m3"
embedding_model = SentenceTransformer(MODEL_NAME, device=device)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

O modelo escolhido foi o modelo BAAI-bge-m3, hoje considerado um dos melhores modelos, senão o melhor, para embedding de tokens. O modelo foi escolhido devido aos seus 3M's:

* Multi-Lingual: Domina e entende mais de 100 idiomas, incluindo inglês e português. O que é muito útil para um texto que mistura conceitos nomeados em inglês e o texto em português.

* Multi-Granularity: suporta até 8.192 tokens de janela.

* Multi-Functionality (Treinado para Semântica + Palavras Exatas): Ele foi treinado simultaneamente para capturar o sentido geral e a correspondência exata de termos/números.

In [4]:
print(f"Dimensão do vetor (Embedding): {embedding_model.get_embedding_dimension()}")
print(f"Limite máximo de tokens por chunk: {embedding_model.max_seq_length}")

Dimensão do vetor (Embedding): 1024
Limite máximo de tokens por chunk: 8192


In [5]:
# carregando dados
df_chunks = pd.read_parquet(PROCESSED_DATA_DIR / "chunks.parquet")
print(f"Total de chunks carregados: {len(df_chunks):,}")
df_chunks.head(2)

Total de chunks carregados: 6,831


,chunk_id,periodo,ano,trimestre,tipo_doc,page_number,chunk_index,char_count,text,text_with_context
0,1T2024_completas_p2_c0,1T2024,2024,1,completas,2,0,897,#### **Índice** \n\n###### **Análise gerencial...,[Relatório Itaú Unibanco | Período: 1T2024 | D...
1,1T2024_completas_p5_c0,1T2024,2024,1,completas,5,0,141,###### Sumário do **Resultado Gerencial** \n\n...,[Relatório Itaú Unibanco | Período: 1T2024 | D...


In [6]:
# pegando os 10 primeiros chunks
sample_texts = df_chunks["text_with_context"].iloc[:10].tolist()

# gerando vetores de embedding
sample_embeddings = embedding_model.encode(
    sample_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"Formato da matriz (chunks, dimensões): {sample_embeddings.shape}")
print(f"Primeiros 10 números do vetor do Chunk 0:\n{sample_embeddings[0][:10]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Formato da matriz (chunks, dimensões): (10, 1024)
Primeiros 10 números do vetor do Chunk 0:
[-0.03752236  0.01356311 -0.02436216 -0.0112041  -0.01826789  0.00670417
 -0.03013602  0.0765091  -0.0027932   0.05156525]


In [7]:
# Definir pergunta de teste e gerar seu vetor
query_text = "Qual foi o Resultado Recorrente Gerencial e o ROE no 1T24?"
query_embedding = embedding_model.encode(query_text, normalize_embeddings=True)

# Calcular a similaridade de cosseno entre a pergunta e os 10 chunks
similarity_scores = embedding_model.similarity(query_embedding, sample_embeddings)[0]


In [8]:
query_embedding

array([-0.08927941,  0.02209692, -0.02357806, ..., -0.01362731,
        0.00284725, -0.00097563], shape=(1024,), dtype=float32)

In [9]:
similarity_scores

tensor([0.5120, 0.4787, 0.6264, 0.5736, 0.4504, 0.5039, 0.4647, 0.5186, 0.5213,
        0.4108])

In [10]:
for idx, score in enumerate(similarity_scores):
    chunk_id = df_chunks.iloc[idx]["chunk_id"]
    text_preview = df_chunks.iloc[idx]["text"][:80].replace("\n", " ")
    print(f"Score: {score:.4f} | {chunk_id:24s} | {text_preview}...")

Score: 0.5120 | 1T2024_completas_p2_c0   | #### **Índice**   ###### **Análise gerencial da operação**   |**Análise gerencia...
Score: 0.4787 | 1T2024_completas_p5_c0   | ###### Sumário do **Resultado Gerencial**   A seguir, apresentamos os indicadore...
Score: 0.6264 | 1T2024_completas_p5_c1   | |**Em R**|**$ milhões (exceto onde indicado)**|**1T24**|**4T23**|**1T23**| |---|...
Score: 0.5736 | 1T2024_completas_p5_c2   | ||Índice de Eficiência (IE) (6)|38,3%|40,3%|39,8%| ||Resultado Recorrente Gerenc...
Score: 0.4504 | 1T2024_completas_p5_c3   | |**Bala**|Índice de Basileia Consolidado Prudencial|16,4%|17,0%|15,0%| ||Índice ...
Score: 0.5039 | 1T2024_completas_p5_c4   | Obs. : (1) O Produto Bancário é a soma da Margem Financeira Gerencial, das Recei...
Score: 0.4647 | 1T2024_completas_p5_c5   | representativas do capital social líquidas das ações em tesouraria; (9) JCP – Ju...
Score: 0.5186 | 1T2024_completas_p6_c0   | ###### Resultado **Gerencial**   Nesse relatório, além do ajuste dos

In [23]:
# verificando o chunk de maior similaridade
texto_chunk = df_chunks.loc[df_chunks['chunk_id'] == '1T2024_completas_p5_c1'].text.values[0]
print(texto_chunk)

|**Em R**|**$ milhões (exceto onde indicado)**|**1T24**|**4T23**|**1T23**|
|---|---|---|---|---|
||Resultado Recorrente Gerencial|9.771|9.401|8.435|
|**DRE**|Produto Bancário (1)|40.353|40.985|37.450|
||Margem Financeira Gerencial (2)|26.880|27.134|24.692|
||Retorno Recorrente Gerencial sobre o Patrimônio Líquido Médio anualizado -Consolidado (3)|21,9%|21,2%|20,7%|
||Retorno Recorrente Gerencial sobre o Patrimônio Líquido Médio anualizado- Brasil (3)|22,7%|22,2%|21,1%|
|**nho**|Retorno Recorrente Gerencial sobre o Ativo Médio anualizado (4)|1,4%|1,4%|1,3%|
|**pe**|Índice de Inadimplência (90 dias) - Total|2,7%|2,8%|2,9%|
|**sem**|Índice de Inadimplência (90 dias) - Brasil|3,1%|3,2%|3,4%|
|**De**|Índice de Inadimplência (90 dias) - América Latina|1,4%|1,4%|1,4%|
||Índice de Cobertura (Saldo de Provisão Total / Operações vencidas há mais de 90 dias) (5)|221%|216%|212%|
||Índice de Eficiência (IE) (6)|38,3%|40,3%|39,8%|
||Resultado Recorrente Gerencial por Ação (R$) (7)|1,00|0,96|0,86|


Com isso, o texto retornado está perfeitamente alinha com a pergunta. Portanto, nossa busca por similaridade parece estar funcionando corretamente.

## 2.2.Testando ChromaDB

In [42]:
# ###### criando a collection
# definindo e criando o path do collection
chroma_path = MODELS_DIR / "vector_store" / "chromadb"
chroma_path.mkdir(parents=True, exist_ok=True)

# criando o cliente chroma
chroma_client = chromadb.PersistentClient(path=str(chroma_path))

# criando um collection de teste
collection_name = "test_chunks"

try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB inicializado em: {chroma_path}")
print(f"Coleção '{chroma_client}' criada com sucesso!")

ChromaDB inicializado em: /home/gabriel_santiago/Estudos/Portfolio/ri_chat/models/vector_store/chromadb
Coleção '<chromadb.api.client.Client object at 0x7c65c98c9810>' criada com sucesso!


In [43]:
collection

Collection(name=test_chunks)

In [37]:
# preparando os dados de testes para
# inserer na collection
sample_df = df_chunks.iloc[:10]
sample_ids = sample_df["chunk_id"].tolist()
sample_docs = sample_df["text"].tolist()
sample_metadatas = [
    {
        "periodo": row.periodo,
        "ano": int(row.ano),
        "trimestre": int(row.trimestre),
        "tipo_doc": str(row.tipo_doc),
        "page_number": int(row.page_number),
    }
    for row in sample_df.itertuples(index=False)
]

In [38]:
sample_metadatas

[{'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 2},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 5},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 6},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_doc': 'completas',
  'page_number': 6},
 {'periodo': '1T2024',
  'ano': 2024,
  'trimestre': 1,
  'tipo_

In [39]:
sample_embeddings

array([[-0.03752236,  0.01356311, -0.02436216, ..., -0.00838777,
        -0.00360223, -0.0199766 ],
       [-0.01389308,  0.0193594 , -0.04700405, ..., -0.0156707 ,
        -0.00122018, -0.06770214],
       [-0.05419531,  0.03093234, -0.00486692, ..., -0.00482395,
        -0.03651442, -0.01278835],
       ...,
       [-0.03856134,  0.0151327 , -0.02102125, ..., -0.00468304,
         0.0065234 , -0.04818281],
       [ 0.00272088,  0.00553659, -0.01856527, ...,  0.00204332,
        -0.02373996,  0.02055969],
       [-0.03597709,  0.00088372, -0.00144604, ..., -0.02053704,
        -0.01078177,  0.01663658]], shape=(10, 1024), dtype=float32)

In [44]:
# inserindo os dados de fato
collection.add(
    ids=sample_ids,
    documents=sample_docs,
    embeddings=sample_embeddings.tolist(),
    metadatas=sample_metadatas,
)

print(f"Total de itens na coleção: {collection.count()}")

Total de itens na coleção: 10


In [45]:
query_text

'Qual foi o Resultado Recorrente Gerencial e o ROE no 1T24?'

In [ ]:
# executando a query filtrando por periodo
query_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2,  # retorna os top 2
    where={"periodo": "1T2024"}  # filtro de metadados: busca apenas no 1T2024!
)

In [ ]:
# exibindo os resultados
print("\n--- Resultados retornados pelo ChromaDB ---")
for i in range(len(query_results["ids"][0])):
    retrieved_id = query_results["ids"][0][i]
    distance = query_results["distances"][0][i]
    meta = query_results["metadatas"][0][i]
    print(f"Rank {i+1} | ID: {retrieved_id} | Cosine Distance: {distance:.4f} | Metadados: {meta}")


--- Resultados retornados pelo ChromaDB ---
Rank 1 | ID: 1T2024_completas_p5_c1 | Cosine Distance: 0.3736 | Metadados: {'periodo': '1T2024', 'page_number': 5, 'trimestre': 1, 'ano': 2024, 'tipo_doc': 'completas'}
Rank 2 | ID: 1T2024_completas_p5_c2 | Cosine Distance: 0.4264 | Metadados: {'tipo_doc': 'completas', 'periodo': '1T2024', 'trimestre': 1, 'ano': 2024, 'page_number': 5}


O ChromaDB não só achou o documento correto, como os dois primeiros resultados cobrem 100% da página que responde a pergunta. Excelente resultado!